[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/optimizing_llm_as_a_judge_prompts.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">优化 LLM 裁判提示词</h1>


**什么是 LLM 裁判？**

LLM 裁判是指将 LLM 用作评估工具，根据预定义标准评判和打分。LLM 充当“裁判”，根据接收到的输入作出判断并分配分数。常见应用包括幻觉、用户挫败感和摘要质量评估。

**为什么要改进这些提示词？**

虽然 LLM 是强大的评估工具，但表现可能不稳定。提示词歧义、模型偏差或缺少明确准则等因素都可能导致结果不可靠。优化 LLM 提示词可以提高模型的一致性、公平性和准确性，让评估更加可靠。

本教程将带你：

- 生成 LLM 裁判评估提示词并在数据集上测试
- 学习多种模板优化技术，每一步都通过 Phoenix 评估衡量准确性
- 理解如何组合这些技术，改进特定场景中的评估

⚠️ 本教程需要 OpenAI 密钥。

开始吧！🚀


# **配置依赖与密钥**


In [ ]:
%pip install -q "arize-phoenix>=8.0.0" datasets openinference-instrumentation-openai

接下来连接 Phoenix。以下代码连接 Phoenix Cloud 实例，也可以选择[连接自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。


In [ ]:
import os
from getpass import getpass

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

# **将数据集加载到 Phoenix**


Phoenix 提供许多用于 LLM 裁判的[内置评估模板](https://arize.com/docs/phoenix/evaluation/concepts-evals/evaluation)，但你可能需要针对特定场景构建自定义评估器。

本教程重点创建评估聊天机器人回复中的共情能力和情商的 LLM 裁判提示词。这对心理健康聊天机器人或客服交互等场景尤其有用。

首先加载包含 30 条聊天机器人回复的数据集，每条都带有共情与情商评分，满分 10 分。教程中将用提示词评估这些回复，并将输出与真实标签比较，以衡量提示词表现。


In [ ]:
import uuid

from datasets import load_dataset
from phoenix.client import Client

px_client = Client()

ds = load_dataset("syeddula/empathy_scores")["test"]
ds = ds.to_pandas()
unique_id = uuid.uuid4()

# 将数据集上传到 Phoenix
dataset = px_client.datasets.create_dataset(
    dataframe=ds, name=f"empathy-{unique_id}", input_keys=["AI_Response", "EI_Empathy_Score"]
)

# **使用元提示生成 LLM 裁判模板**


迭代模板之前，需要先建立提示词。运行以下单元，将生成专门评估共情能力和情商的 LLM 裁判提示词。生成模板时强调：

- 选择评估标准，例如共情、情感支持、情商。
- 定义清晰的评分体系：1～10 分，并说明各分数含义。
- 制定回复格式准则，确保清晰和一致。
- 解释 LLM 选择该分数的原因。


In [ ]:
from openai import OpenAI

client = OpenAI()


def generate_eval_template():
    meta_prompt = """
    你是 AI 评估和情商评估专家。你的任务是创建结构化评估模板，评判 AI 针对用户输入的回复所体现的情商与共情能力。

    ### 任务概述：
    生成详细模板，衡量 AI 识别用户情绪、表达共情并给出情感恰当回复的能力。模板应：
    - 强调给定的 AI 输入正是待评估对象
    - 包含不同评估标准，衡量情商的不同方面
    - 定义 1～10 分的评分体系，使不同回复获得广泛分布的评分
    - 提供明确的分层评分准则，区分较弱、一般与优秀表现
    - 包含理由部分，要求评估者用具体示例解释分数

    ### 格式：
    以纯文本返回评估模板，包含标题、标准和详细评分细则。模板应易于理解并适用于真实数据集。

    ### 变量：开头应留出传入待评估 AI 回复的位置，格式必须为：{input}。明确说明这是被评估的输入。

    ### 评分准则：
    - 必须使用 **1～10 分**，鼓励使用完整分数范围。强调整体评估输出只能是 1～10 的一个数字。
    - 使用具体、清晰的等级区分优秀、一般和较弱回复。
    - 要求评估者解释评分。

    不要添加“模板结束”等结语，模板应在最后一个部分后自然结束。

    限 175 个词。

    """

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": meta_prompt}],
            temperature=0.9,  # 提高温度以增加创造性
        )

        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}


print("正在生成新的评估模板……")
EMPATHY_EVALUATION_PROMPT_TEMPLATE = generate_eval_template()
print("模板生成成功！")
print(EMPATHY_EVALUATION_PROMPT_TEMPLATE)

# **测试初始提示词**


为应用插桩，将追踪发送到 Phoenix：


In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="LLM-as-a-Judge", auto_instrument=True)

已有基线提示词后，需要设置两个关键组件：

- **任务**：LLM 裁判评估，由模型根据共情能力和情商为聊天机器人回复评分。
- **评估器**：将 LLM 裁判输出与数据集真实标签比较的函数。

最后运行实验。通过此配置可以衡量初始提示词的表现。


In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator, bind_evaluator, create_evaluator

llm = LLM(provider="openai", model="gpt-5")

empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: row["AI_Response"]},
)


# 实验任务：对数据集每行运行共情评估器
def run_empathy_eval(input):
    result = empathy_evaluator.evaluate(input)
    return result[0]


# 实验评估器
@create_evaluator(name="close_match")
def close_match(output, input):
    expected_score = int(input["EI_Empathy_Score"])
    predicted_score = int(output["label"]) if output else -1
    return abs(expected_score - predicted_score) <= 2

In [ ]:
from phoenix.client import AsyncClient

async_client = AsyncClient()

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="baseline",
    evaluators=[close_match],
)

# **第 1 次迭代：提高准确性**


如果 LLM 裁判提示词准确性较低，可以通过调整提示词改进。本节探索两种技术：少样本示例和人工参与。


## 技术 1：少样本示例


少样本示例通过提供清晰的评估参照来提高 LLM 裁判提示词的准确性。模型不再只依赖一般指令，而是从展示正确评分和理由的标注示例中学习。

混合使用高、中、低分回复，能够帮助模型：

- 理解共情能力和情商等细微标准。
- 与现实判断对齐，减少不一致。


In [ ]:
few_shot_examples = """
---
为帮助推理，将文本与以下示例及评分比较：

示例 1："You should try to focus and get your work done. Everyone has tough days."
  Score: 2
  解释：未认可用户情绪，态度敷衍，没有提供真正支持。

示例 2："That sounds really stressful. Have you considered talking to a supervisor or taking breaks?"
  Score: 5
  解释：认可了压力，但比较泛泛。提供了建议，但不够个性化，语气可以更温暖。

示例 3："I’m really sorry you’re feeling this way. It’s completely understandable to feel overwhelmed. You’re not alone in this. Have you had a chance to take a break or talk to someone who can support you?"
  Score: 9
  解释：认可情绪、安慰用户并提供支持。
"""
EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples

In [ ]:
empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: [row["AI_Response"]]},
)

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="few-shot",
    evaluators=[close_match],
)

## 技术 2：人工参与


人工参与通过监督、验证和必要时纠正，提高 LLM 裁判的准确性。在 Phoenix 中可通过[标注](https://arize.com/docs/phoenix/tracing/features-tracing/how-to-annotate-traces)实现。LLM 按预定义标准评估回复，人工审核则可以：

- 发现模型可能遗漏的边界情况与偏差。
- 识别 LLM 输出中的不一致，完善评分准则。
- 分析模型困难之处并相应调整指令，持续改进提示词。

不过，人工审核成本高且耗时，全面标注可能不现实。好在即使少量人工标注示例也能显著提高准确性。


![人工标注](https://storage.googleapis.com/arize-phoenix-assets/assets/images/human_annotation.gif)


# **第 2 次迭代：减少偏差**


## 风格无关评估


LLM 裁判评估的一种常见偏差是偏爱某些写作风格。例如，即使两份回复表现出相同的共情或情商水平，模型也可能无意中给正式、结构化的回复打出高于随意或简洁回复的分数。

为减少偏差，我们采用风格无关评估，确保 LLM 根据内容而非措辞或语气评判回复。具体方法：

- 提供涵盖不同写作风格的多样化少样本示例。
- 评估不同措辞的回复以测试偏差，确保评分一致。

让评估不依赖风格，可以建立更稳健的评分体系，避免无意中惩罚某些语气。


In [ ]:
style_invariant = """
----
为帮助推理，下面展示不同回复风格和语气如何获得相近评分：

#### 场景：客服处理延迟订单
用户："My order is late, and I needed it for an important event. This is really frustrating."

回复 A（正式）："I sincerely apologize for the delay..."
回复 B（随意）："Oh no, that’s really frustrating!..."
回复 C（直接）："Sorry about that. I’ll check..."
"""
EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + style_invariant

In [ ]:
empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: [row["AI_Response"]]},
)

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="style invariant",
    evaluators=[close_match],
)

# **第 3 次迭代：降低成本与延迟**


较长的提示词会增加计算成本与响应时间，使评估更慢、更贵。为提高效率，我们在保持清晰和有效的前提下压缩提示词：

- 删除冗余指令并简化措辞。
- 使用项目符号或结构化格式提供简洁指导。
- 保留关键评估标准，删除不必要的解释。

优化良好的提示词可减少 token 数，在不牺牲准确性和可靠性的前提下，提高评估速度并降低成本。


In [ ]:
def generate_condensed_template():
    meta_prompt = """
    你是提示词工程和 LLM 评估专家。你的任务是优化给定的 LLM 裁判提示词，显著减少词数，同时保留评估标准、评分体系和目的等全部必要信息。

    要求：
    保留指标、评分准则和判断标准等关键细节。

    删除冗余措辞和不必要解释。

    在不丢失含义的前提下保持清晰简洁。

    保持提示词在一致评估方面的有效性。

    输出格式：
    仅以纯文本返回优化后的提示词，不要解释或评论。

    """

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {
                    "role": "user",
                    "content": "提供的 LLM 裁判提示词"
                    + EMPATHY_EVALUATION_PROMPT_TEMPLATE,
                },
                {"role": "user", "content": meta_prompt},
            ],
            temperature=0.9,  # 提高温度以增加创造性
        )

        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}


print("正在生成压缩后的评估模板……")
EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED = generate_condensed_template()
print("模板生成成功！")
print(EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED)

In [ ]:
empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: [row["AI_Response"]]},
)

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="condensed",
    evaluators=[close_match],
)

# **第 4 次迭代：自我改进（迭代式 LLM 裁判）**


自我改进让裁判通过批判性分析和调整初始判断来改善评估。模型不再只给出固定分数，而是迭代执行：

- 根据评估标准生成初始分数。
- 反思推理，检查不一致或偏差。
- 必要时修正分数，确保符合评估准则。

采用这种推理方式后，模型可以解释判断依据并自行纠错。


In [ ]:
refinement_text = """
---
完成评估后，执行以下两步：
1. 自我审视
检查初始分数：
- 是否过于严格或宽松？
- 是否考虑完整上下文？
- 其他人是否会认可你的评分？
简要解释任何不一致。

2. 最终完善
根据审视结果，必要时调整分数。
- 仅输出一个数字（1～10）
"""
EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + refinement_text

In [ ]:
empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: [row["AI_Response"]]},
)

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="self-refinement",
    evaluators=[close_match],
)

# **第 5 次迭代：组合多种技术**


为最大限度提高裁判的准确性与公平性，我们组合多种优化技术。本例结合少样本示例和风格无关评估，确保模型关注内容而非措辞或语气。

通过组合这些技术，建立更可靠的评估框架。


In [ ]:
EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = (
    EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples + style_invariant
)

In [ ]:
empathy = ClassificationEvaluator(
    name="empathy",
    llm=llm,
    prompt_template=EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED,
    choices=list(map(str, range(1, 11))),
)

empathy_evaluator = bind_evaluator(
    empathy,
    input_mapping={"input": lambda row: [row["AI_Response"]]},
)

experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=run_empathy_eval,
    experiment_name="combined",
    evaluators=[close_match],
)

# **最终结果**


少样本示例、自我改进、风格无关评估和提示词压缩等技术各有优势，但效果因任务而异。

**注意**：有时会看到性能下降，这未必是“错误”。LLM 选择及其他模型固有行为都会影响结果。

通过系统测试并组合这些方法，可以逐步完善评估框架。


![结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/llm_as_a_judge_tutorial_results.png)


接下来可以查看更多 [Phoenix 示例](https://arize.com/docs/phoenix/notebooks)，也欢迎[在 GitHub 上为我们点星！](https://github.com/Arize-ai/phoenix) ⭐️
